In [0]:
import requests
import json
import time

print("Libraries loaded successfully")

In [0]:
indicators = {
    "NY.GDP.MKTP.CD": "GDP (current US$)",
    "NY.GDP.PCAP.CD": "GDP per capita",
    "SP.POP.TOTL": "Population, total",
    "SI.POV.DDAY": "Poverty headcount ($2.15/day)",
    "SP.DYN.LE00.IN": "Life expectancy at birth",
    "SE.ADT.LITR.ZS": "Literacy rate",
    "SH.XPD.CHEX.GD.ZS": "Health expenditure (% GDP)",
    "EN.ATM.CO2E.PC": "CO2 emissions per capita",
    "FP.CPI.TOTL.ZG": "Inflation (%)",
    "SL.UEM.TOTL.ZS": "Unemployment (%)"
}

print(f"Total indicators selected: {len(indicators)}")

In [0]:
def pull_indicator(indicator_code, start_year, end_year):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
    params = {
        "format": "json",
        "date": f"{start_year}:{end_year}",
        "per_page": 20000
    }
    response = requests.get(url, params=params)
    data = response.json()
    if len(data) > 1 and data[1]:
        return data[1]
    return []

print("Function ready")

In [0]:
full_load_data = []

for code, name in indicators.items():
    print(f"Pulling: {name} ({code})...")
    records = pull_indicator(code, 1990, 2024)
    full_load_data.extend(records)
    time.sleep(0.5)

print(f"\nTotal records pulled: {len(full_load_data)}")

In [0]:
# Sirf first 2000 records sample ke liye lo (poora data already tumhare paas hai memory mein)
full_load_sample = full_load_data[:2000]

with open("/tmp/full_load_sample.json", "w") as f:
    json.dump(full_load_sample, f, indent=2)

print(f"Sample saved with {len(full_load_sample)} records")

In [0]:
import base64

with open("/tmp/full_load_sample.json", "rb") as f:
    content = f.read()

b64 = base64.b64encode(content).decode()
html = f'<a download="full_load_sample.json" href="data:application/json;base64,{b64}">Click here to download full_load_sample.json</a>'
displayHTML(html)

In [0]:
incremental_data = []

for code, name in indicators.items():
    print(f"Pulling recent: {name} ({code})...")
    records = pull_indicator(code, 2023, 2024)
    incremental_data.extend(records)
    time.sleep(0.5)

print(f"\nIncremental records pulled: {len(incremental_data)}")

In [0]:
incremental_sample = incremental_data[:1000]

with open("/tmp/incremental_load_sample.json", "w") as f:
    json.dump(incremental_sample, f, indent=2)

print(f"Incremental sample saved with {len(incremental_sample)} records")

In [0]:
with open("/tmp/incremental_load_sample.json", "rb") as f:
    content = f.read()

b64 = base64.b64encode(content).decode()
html = f'<a download="incremental_load_sample.json" href="data:application/json;base64,{b64}">Click here to download incremental_load_sample.json</a>'
displayHTML(html)

In [0]:
import json

# Poora full_load_data save karo — koi truncate nahi
with open("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json", "w") as f:
    json.dump(full_load_data, f, indent=2)

print(f"Saved {len(full_load_data)} records")

In [0]:
with open("/Workspace/Users/l242585@lhr.nu.edu.pk/incremental_full.json", "w") as f:
    json.dump(incremental_data, f, indent=2)

print(f"Saved {len(incremental_data)} records")

In [0]:
import os
size_mb = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json") / (1024*1024)
print(f"File size: {size_mb:.2f} MB")

In [0]:
import requests
import time

# World Bank ke sab topics (categories) list karo
topics_resp = requests.get("https://api.worldbank.org/v2/topic?format=json").json()
topics = topics_resp[1]

print(f"Total topics found: {len(topics)}")

# Har topic se kuch indicators uthao jab tak ~200 tak na pahunch jayein
indicators = {}
target_count = 200

for t in topics:
    if len(indicators) >= target_count:
        break
    topic_id = t['id']
    url = f"https://api.worldbank.org/v2/topic/{topic_id}/indicator"
    params = {"format": "json", "per_page": 15}  # har topic se 15 indicators
    resp = requests.get(url, params=params).json()
    
    if len(resp) > 1 and resp[1]:
        for ind in resp[1]:
            if len(indicators) < target_count:
                indicators[ind['id']] = ind['name']
    time.sleep(0.3)

print(f"Total indicators collected: {len(indicators)}")

In [0]:
full_load_data = []

for code, name in indicators.items():
    print(f"Pulling: {name} ({code})...")
    records = pull_indicator(code, 1960, 2024)   # 1990 ki jagah 1960
    full_load_data.extend(records)
    time.sleep(0.3)

print(f"\nTotal records pulled: {len(full_load_data)}")

In [0]:
incremental_data = []

for code, name in indicators.items():
    print(f"Pulling recent: {name} ({code})...")
    records = pull_indicator(code, 2023, 2024)
    incremental_data.extend(records)
    time.sleep(0.3)

print(f"\nIncremental records pulled: {len(incremental_data)}")

In [0]:
import json

# Apna email daalo yahan (Databricks login wala)
with open("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json", "w") as f:
    json.dump(full_load_data, f, indent=2)

print(f"Saved {len(full_load_data)} records")

In [0]:
with open("/Workspace/Users/l242585@lhr.nu.edu.pk/incremental_full.json", "w") as f:
    json.dump(incremental_data, f, indent=2)

print(f"Saved {len(incremental_data)} records")

In [0]:
import os

full_size = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json") / (1024*1024)
inc_size = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/incremental_full.json") / (1024*1024)

print(f"Full load size: {full_size:.2f} MB")
print(f"Incremental size: {inc_size:.2f} MB")

In [0]:
import json

# indent=2 hatao — compact JSON save karo
with open("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json", "w") as f:
    json.dump(full_load_data, f)   # indent parameter nahi diya

print("Saved (compact format)")

In [0]:
import os

# Current file se bytes-per-record nikalo
current_size_bytes = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json")
bytes_per_record = current_size_bytes / len(full_load_data)

# Kitne total records chahiye 200MB ke liye
target_bytes = 200 * 1024 * 1024
target_records = target_bytes / bytes_per_record

# Kitne indicators chahiye (proportion se)
current_indicator_count = len(indicators)
needed_indicator_count = int(current_indicator_count * (target_records / len(full_load_data)))

print(f"Current: {current_indicator_count} indicators → {len(full_load_data)} records → {current_size_bytes/(1024*1024):.1f} MB")
print(f"Target: ~{needed_indicator_count} indicators needed for ~200 MB")

In [0]:
# indicators dictionary ko chote target ke hisaab se trim karo
trimmed_indicators = dict(list(indicators.items())[:needed_indicator_count])

full_load_data = []
for code, name in trimmed_indicators.items():
    print(f"Pulling: {name} ({code})...")
    records = pull_indicator(code, 1960, 2024)
    full_load_data.extend(records)
    time.sleep(0.3)

print(f"\nTotal records: {len(full_load_data)}")

# Compact format mein save karo
with open("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json", "w") as f:
    json.dump(full_load_data, f)

size_mb = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json") / (1024*1024)
print(f"New file size: {size_mb:.2f} MB")

In [0]:
# Top 70 indicators select karo
trimmed_indicators = dict(list(indicators.items())[:70])

print(f"Using {len(trimmed_indicators)} indicators")

full_load_data = []
for code, name in trimmed_indicators.items():
    print(f"Pulling: {name} ({code})...")
    records = pull_indicator(code, 1960, 2024)
    full_load_data.extend(records)
    time.sleep(0.3)

print(f"\nTotal records: {len(full_load_data)}")

In [0]:
import json
import os

with open("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json", "w") as f:
    json.dump(full_load_data, f)

size_mb = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json") / (1024*1024)
print(f"Full load file size: {size_mb:.2f} MB")

In [0]:
incremental_data = []
for code, name in trimmed_indicators.items():
    print(f"Pulling recent: {name} ({code})...")
    records = pull_indicator(code, 2023, 2024)
    incremental_data.extend(records)
    time.sleep(0.3)

with open("/Workspace/Users/l242585@lhr.nu.edu.pk/incremental_full.json", "w") as f:
    json.dump(incremental_data, f)

inc_size_mb = os.path.getsize("/Workspace/Users/l242585@lhr.nu.edu.pk/incremental_full.json") / (1024*1024)
print(f"Incremental records: {len(incremental_data)}")
print(f"Incremental file size: {inc_size_mb:.2f} MB")

In [0]:
import json, os

src = "/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_full.json"
with open(src) as f:
    data = json.load(f)

parts = 3
chunk = -(-len(data) // parts)  # ceil division

for i in range(parts):
    part = data[i*chunk:(i+1)*chunk]
    path = f"/Workspace/Users/l242585@lhr.nu.edu.pk/full_load_part{i+1}.json"
    with open(path, "w") as f:
        json.dump(part, f)
    print(path, round(os.path.getsize(path)/(1024*1024), 1), "MB")

In [0]:
import json, os

base = "/Workspace/Users/l242585@lhr.nu.edu.pk"

# Full load: teeno parts ka count aur size
full_total = 0
full_mb = 0
for i in (1, 2, 3):
    path = f"{base}/full_load_part{i}.json"
    with open(path) as f:
        n = len(json.load(f))
    mb = os.path.getsize(path) / (1024 * 1024)
    full_total += n
    full_mb += mb
    print(f"part{i}: {n:,} records, {mb:.2f} MB")

print(f"\nFULL LOAD TOTAL: {full_total:,} records, {full_mb:.2f} MB")

# Incremental load
inc_path = f"{base}/incremental_full.json"
with open(inc_path) as f:
    inc = json.load(f)
print(f"INCREMENTAL: {len(inc):,} records, {os.path.getsize(inc_path)/(1024*1024):.2f} MB")